# CwH Stage 2 — narrow pattern sweep (Tier A) x wide exit sweep (Tier B), PIT_C base

**LOCKBOX:** every series is cut at **2024-12-31 at load**; IS = entries **2011-01-01 .. 2023-12-29**
(purge covers the longest 250-bar time stop; prices through 2024-12-31 let late-2023 entries resolve).
13 IS years = the CPCV groups. 2025+ is never loaded.

**Provenance (Step 0 audit):** no PIT_C parameter has a literature source. Fixed-but-unsourced (not swept):
rim match 15%, pivots 5/5, rounded-bottom test, 30-bar handle window, 15-bar fire window, v2b 50>200.
Re-opened: cup length, cup depth, handle rules, stop, target, time stop, entry. JACK-fitted on look-ahead
data (not used here): MAX_HANDLE_DAYS / staleness, handle_score.

**Tier A (one candidate-cup enumeration, then per-variant rules; one cup per left lip, as PIT_C):**
A1 PIT_C control · A2 O'Neil "commonly cited" (cup 35-325 bars, depth 12-33%, handle low in upper half of
the base, handle drop <= 15% from the right lip, prior uptrend >= 30% over 252 bars into the left lip;
"handle >= 5 bars" is automatically true — the right lip is a strict pivot) · A3 dropped (not sourceable) ·
A4 cup >= 60 bars · A5 depth <= 33% · A6 handle low in upper half of the base (Tier B only if >= 10% of its
next-open fires differ from A1's — decided on fire counts, before any outcome).

**v2b is evaluated at the DECISION bar** (fire bar j for next-open, t-1 for buy-stop). PIT_C evaluated it
at the entry bar, using that bar's close — a one-bar look-ahead, kept ONLY in the A1 parity check.

**Tier B per variant (2 x 5 x 8 x 5 x 3 x 2 = 2,400 configs):** entry {next_open, buy_stop} · stop
{handle (PIT_C rule), 7%, 8%, 2xATR, 3xATR} · target {t05, t075, t10, t15 (rim + k x cup depth),
+15%, +20%, +25%, none} · trail {none, close<SMA10/21/50, close < highest-high-since-entry - 3xATR} ·
time {60, 120, 250} · regime gate {off, on}. No pruning (every combo exits via stop or time).
**Max initial stop distance 10% of the fill applies to ALL stop types** (trade skipped otherwise).
Fills are realistic: a gap through the stop fills at min(open, stop); through the target at max(open, target).
Costs: 0.15% of price round trip -> R cost = 0.0015 x fill / (fill - stop). Non-overlap per config.

**GATES (fixed before any result):**
1. PF_cost >= 1.30 · month-block bootstrap 90% CI lower bound on PF_cost > 1.0 · n >= 300
2. yearly avg % return (after cost) beats the A1 control in >= 9 of 13 IS years (a year with no trades = 0%)
   · no single year > 25% of net R (after cost)
3. PLATEAU: median PF_cost of one-step neighbours >= 1.20 (undefined PF counts as 0). Ordered dims ->
   adjacent values only: target t05-t075-t10-t15-none and +15%-+20%-+25%-none; stop ordered by IS median
   initial stop distance (A1 next-open fires, computed from geometry before any exit is simulated); trail
   SMA10-SMA21-SMA50-none-chandelier; time 60-120-250. Categorical dims -> all alternatives: entry, gate.
4. PBO < 0.20: CPCV, groups = the 13 IS years, all C(13,6) = 1,716 splits with 6 test years; select max
   PF_cost on the 7 train years among configs with >= 100 train trades; omega = OOS rank / (N+1); PBO = P(logit omega <= 0).
   Over the FULL Stage 2 matrix.
5. DSR > 0.95 on EFFECTIVE N. Series = monthly cost-adjusted R (156 IS months, empty = 0); SR = mean/std
   (ddof=1); Pearson kurtosis; Var[SR] across Stage 2 configs. Effective N = clusters from hierarchical
   agglomerative clustering, AVERAGE linkage, distance = 1 - Pearson rho of those monthly series, cut at
   distance 0.50, over ALL logged trials (Stage 1's 212 + Stage 2); zero-variance series = own cluster.
   Raw-N DSR (N = 212 + Stage 2 count) reported as sensitivity.
Survivors (pass all five) ranked by DSR; at most 3 go to a separate final notebook.

**Resumable:** every expensive cell saves to `results/stage2_PIT_C/cache/` and skips work already saved.
Runtime estimates are in each cell's header comment.


In [ ]:
# ===== Cell 0 — Config, gates, dimensions (prints the gates before anything runs) =====
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, glob, time, json, hashlib, os, itertools, math
from pathlib import Path
from numba import njit, prange
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/Bukowski')
HIST_DIR = DATA_DIR/'results'/'history_15yr'          # READ-ONLY: PIT_C outputs (A1 parity)
S1_DIR   = DATA_DIR/'results'/'sweep_PIT_C'           # the append-only trial log lives here
REF_DIR  = DATA_DIR/'reference'
OUT_DIR  = DATA_DIR/'results'/'stage2_PIT_C'; CACHE = OUT_DIR/'cache'
CACHE.mkdir(parents=True, exist_ok=True)
RSP_PATH = DATA_DIR/'RSP.csv' if (DATA_DIR/'RSP.csv').exists() else REF_DIR/'RSP.csv'

# ---- LOCKBOX ----
CUT_DATE   = pd.Timestamp('2024-12-31')     # every series cut at load
PURGE_DATE = pd.Timestamp('2023-12-29')     # IS entries <= this (covers the 250-bar time stop)
IS_START   = pd.Timestamp('2011-01-01')
IS_YEARS   = list(range(2011, 2024))         # 13 CPCV groups
N_MONTHS   = 12 * len(IS_YEARS)

# ---- PIT_C machinery (fixed, unsourced) ----
ATR_PERIOD, STOCK_FAST_SMA, STOCK_SLOW_SMA = 14, 50, 200
PIV, RIM_TOL, ROUND_MIN, ROUND_BAND, ROUND_SPAN = 5, 0.15, 3, 0.05, 5
HANDLE_WINDOW, CONFIRM_WINDOW_BARS = 30, 15
MIN_STOP_ATR, MAX_STOP_PCT = 1.0, 0.10
ENUM_DUR, ENUM_DEPTH = (30, 325), (0.12, 0.50)    # widest bounds over all variants
TIME_STOP_DAYS = 120                              # PIT_C (parity only)

VARIANTS = {
    'A1': dict(dur=(30, 130),  depth=(0.12, 0.50), handle='pitc'),
    'A2': dict(dur=(35, 325),  depth=(0.12, 0.33), handle='oneil', handle_max_drop=0.15, prior_up=0.30, prior_lb=252),
    'A4': dict(dur=(60, 130),  depth=(0.12, 0.50), handle='pitc'),
    'A5': dict(dur=(30, 130),  depth=(0.12, 0.33), handle='pitc'),
    'A6': dict(dur=(30, 130),  depth=(0.12, 0.50), handle='pitc_upper'),
}
A6_MIN_DIFF = 0.10

# ---- Tier B dimensions ----
ENTRIES = ['next_open', 'buy_stop']
STOPS   = ['handle', 'pct7', 'pct8', 'atr2', 'atr3']
TARGETS = ['t05', 't075', 't10', 't15', 'p15', 'p20', 'p25', 'none']
TRAILS  = ['none', 'sma10', 'sma21', 'sma50', 'chand3']
TIMES   = [60, 120, 250]
GATES   = ['off', 'on']
TARGET_NBRS = {'t05': ['t075'], 't075': ['t05', 't10'], 't10': ['t075', 't15'], 't15': ['t10', 'none'],
               'p15': ['p20'], 'p20': ['p15', 'p25'], 'p25': ['p20', 'none'], 'none': ['t15', 'p25']}
TRAIL_NBRS  = {'sma10': ['sma21'], 'sma21': ['sma10', 'sma50'], 'sma50': ['sma21', 'none'],
               'none': ['sma50', 'chand3'], 'chand3': ['none']}
TIME_NBRS   = {60: [120], 120: [60, 250], 250: [120]}
CONTROL = dict(variant='A1', entry='next_open', stop='handle', target='t05', trail='none', time=120, gate='off')
COST_PCT, CHAND_MULT = 0.0015, 3.0

GATE = dict(min_pf_cost=1.30, min_ci_lo=1.0, ci_level=0.90, boot_reps=2000, min_n=300,
            min_years_beat=9, n_is_years=13, max_year_share=0.25, plateau_median_pf=1.20,
            max_pbo=0.20, min_dsr=0.95)
PBO_CFG  = dict(groups='IS years 2011-2023', n_test=6, metric='PF_cost', min_n_train=100)
EFFN_CFG = dict(method='hierarchical agglomerative', linkage='average', distance='1 - Pearson rho, monthly cost-adjusted R',
                threshold=0.50, population='Stage 1 (212) + Stage 2', zero_variance='own cluster')
DSR_CFG  = dict(series='monthly cost-adjusted R, 156 IS months, empty = 0', sr='mean/std ddof=1',
                kurtosis='Pearson', var_sr='across Stage 2 configs', n_gate='effective', n_sensitivity='raw')
SWEEP_ID = 'PIT_C_stage2_v1'
BUILT_FROM_COMMIT = '1105333e2c55fdc5bc28bf57f8390621ac78428b'
STAGE1_TRIALS = 212

print('LOCKBOX: cut', CUT_DATE.date(), '| IS entries', IS_START.date(), '..', PURGE_DATE.date(), f'| {len(IS_YEARS)} IS years')
print('GATES (fixed):', json.dumps(GATE)); print('PBO:', PBO_CFG); print('EFFECTIVE N:', EFFN_CFG); print('DSR:', DSR_CFG)
print('variants:', list(VARIANTS), '| Tier B per variant:', len(ENTRIES)*len(STOPS)*len(TARGETS)*len(TRAILS)*len(TIMES)*len(GATES))


In [ ]:
# ===== Cell 1 — Load prices CUT AT LOAD, trailing indicators, regime series  (~5-8 min) =====
def compute_atr(df, period=14):
    h,l,c = df['High'],df['Low'],df['Close']; pc=c.shift(1)
    tr = pd.concat([(h-l),(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    return tr.rolling(period).mean()

def load_px(path):
    """Read OHLC(+V), CUT at CUT_DATE before anything else touches it. Returns (cut_df, n_full_rows).
    n_full_rows (a row COUNT only) keeps the >=260-bar universe rule identical to PIT_C."""
    cols = pd.read_csv(path, nrows=0).columns
    use = ['Date','Open','High','Low','Close'] + (['Volume'] if 'Volume' in cols else [])
    d = pd.read_csv(path, usecols=use); d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
    n_full = len(d)
    d = d[d['Date'] <= CUT_DATE].reset_index(drop=True)                  # LOCKBOX
    if 'Volume' not in d.columns: d['Volume'] = np.nan
    return d, n_full

def to_arrays(d):
    d = d.copy()
    d['ATR']  = compute_atr(d, ATR_PERIOD)
    d['S50']  = d['Close'].rolling(STOCK_FAST_SMA).mean()
    d['S200'] = d['Close'].rolling(STOCK_SLOW_SMA).mean()
    return dict(Date=d['Date'].values, Open=d['Open'].to_numpy(float), High=d['High'].to_numpy(float),
                Low=d['Low'].to_numpy(float), Close=d['Close'].to_numpy(float), Volume=d['Volume'].to_numpy(float),
                ATR=d['ATR'].to_numpy(float), S50=d['S50'].to_numpy(float), S200=d['S200'].to_numpy(float))

def read_is_only(path, cols):
    """Read only rows with entry_date <= PURGE_DATE (dates first, every other row skipped), so no
    post-purge row is ever loaded. Row order of the file is preserved."""
    dts = pd.to_datetime(pd.read_csv(path, usecols=['entry_date'])['entry_date'])
    keep = set((dts.index[dts <= PURGE_DATE] + 1).tolist())          # +1: header is line 0
    date_cols = [c for c in cols if c.endswith('_date')]
    return pd.read_csv(path, usecols=cols, parse_dates=date_cols,
                       skiprows=lambda i: i > 0 and i not in keep)[cols]

nan = np.nan
def _idx_le(dates, d):
    """last index with date <= d (-1 if none)"""
    return int(np.searchsorted(dates, np.datetime64(d), side='right')) - 1

def mkt_features(d, spy, rsp):
    out = dict(spy_gt_200=nan, spy_50_gt_200=nan, rsp_spy_20d=nan)
    k = _idx_le(spy['Date'], d); c = spy['Close']
    if k >= 199:
        s200 = c[k-199:k+1].mean()
        out['spy_gt_200'] = float(c[k] > s200)
        out['spy_50_gt_200'] = float(c[k-49:k+1].mean() > s200)
    r = _idx_le(rsp['Date'], d)
    if k >= 20 and r >= 20:
        out['rsp_spy_20d'] = (rsp['Close'][r]/rsp['Close'][r-20] - 1) - (c[k]/c[k-20] - 1)
    return out


def breadth_window(W):
    """THE breadth definition — the ONLY implementation, called by both the fast path and the cut-off
    recompute on bitwise-identical inputs. W = the last <=50 rows (dates <= d) x tickers, last row = d.
    Eligible on d: a close on d AND 50 non-missing closes in the 50 rows ending at d (only bars <= d).
    Returns (breadth, n_eligible). One helper on one array layout -> identical floats; pandas'
    rolling mean returns a flat window EXACTLY while a direct mean can land 1 ulp low, which flipped
    close > SMA50 on stale/flat stretches (the 2011-12 mismatches)."""
    W = np.ascontiguousarray(W, dtype=float)
    if W.shape[0] < 50: return nan, 0
    ok = ~np.isnan(W).any(axis=0)
    sma = W.sum(axis=0) / 50.0
    n_el = int(ok.sum())
    return (float((W[-1] > sma)[ok].sum()) / n_el if n_el else nan), n_el

def build_panels(src, cal):
    """Universe panels on the SPY calendar. Row d depends only on rows <= d."""
    close = pd.DataFrame({tk: pd.Series(p['Close'], index=pd.DatetimeIndex(p['Date'])) for tk, p in src.items()})
    close = close.reindex(pd.DatetimeIndex(cal))
    A = np.ascontiguousarray(close.to_numpy(float))
    bn = [breadth_window(A[max(0, t-49):t+1]) for t in range(len(A))]
    breadth = pd.Series([b for b, _ in bn], index=close.index)
    n_elig = pd.Series([n for _, n in bn], index=close.index)
    rs6  = (close / close.shift(126) - 1).rank(axis=1, pct=True)
    rs12 = (close / close.shift(252) - 1).rank(axis=1, pct=True)
    return dict(close=close, breadth=breadth, n_elig=n_elig, rs6=rs6, rs12=rs12)


def add_trail_arrays(p):
    c = pd.Series(p['Close'])
    for k in (10, 21, 50): p[f'SMA{k}'] = c.rolling(k).mean().to_numpy(float)
    return p

def regime_series(panels, spy, rsp, cal):
    """Daily market regime on the SPY calendar: breadth_50 (shared Stage 1 breadth_window) and
    rsp_spy_20d (Stage 1 mkt_features), both from bars <= the date only."""
    rows = [mkt_features(pd.Timestamp(d), spy, rsp)['rsp_spy_20d'] for d in cal]
    return pd.DataFrame({'breadth_50': panels['breadth'].to_numpy(float), 'rsp_spy_20d': rows},
                        index=pd.DatetimeIndex(cal))
# ---- run ----
files = sorted(f for f in glob.glob(f'{DATA_DIR}/*.csv') if Path(f).stem.upper() != 'SPY')
price_db = {}; N_FULL = {}; PANEL_SRC = {}; t0 = time.time()
for i, f in enumerate(files):
    tk = Path(f).stem.upper()
    try:
        d, n_full = load_px(f)
        if len(d) == 0: continue
        PANEL_SRC[tk] = dict(Date=d['Date'].values, Close=d['Close'].to_numpy(float))
        if n_full < 260: continue
        price_db[tk] = add_trail_arrays(to_arrays(d)); N_FULL[tk] = n_full
    except Exception:
        pass
    if (i+1) % 400 == 0: print(f'  loaded {len(price_db)} / {i+1} ({time.time()-t0:.0f}s)')
spy_df, _ = load_px(DATA_DIR/'SPY.csv'); rsp_df, _ = load_px(RSP_PATH)
SPY = dict(Date=spy_df['Date'].values, Close=spy_df['Close'].to_numpy(float))
RSP = dict(Date=rsp_df['Date'].values, Close=rsp_df['Close'].to_numpy(float))
assert max(v['Date'][-1] for v in price_db.values()) <= np.datetime64(CUT_DATE)
REG_PATH = CACHE/'regime.parquet'
if REG_PATH.exists():
    REGIME = pd.read_parquet(REG_PATH)
else:
    REGIME = regime_series(build_panels(PANEL_SRC, SPY['Date']), SPY, RSP, SPY['Date']); REGIME.to_parquet(REG_PATH)
_isd = REGIME[(REGIME.index >= IS_START) & (REGIME.index <= PURGE_DATE)]
REG_EDGES = {k: float(_isd[k].quantile(0.80)) for k in ['breadth_50', 'rsp_spy_20d']}   # IS trading days only
REGIME['on'] = (REGIME.breadth_50 > REG_EDGES['breadth_50']) & (REGIME.rsp_spy_20d > REG_EDGES['rsp_spy_20d'])
print(f'Universe {len(price_db)} | panel {len(PANEL_SRC)} | regime top-quintile edges (IS days): {REG_EDGES} '
      f'| gate-on share of IS days {REGIME.loc[_isd.index, "on"].mean():.3f} | {time.time()-t0:.0f}s')


In [ ]:
# ===== Cell 2 — Candidate-cup enumeration, widest bounds  (~45-90 min; resumable, 200-ticker chunks) =====
def find_pivot_highs(H, left=5, right=5):
    n=len(H); out=[]
    for i in range(left, n-right):
        w=H[i-left:i+right+1]
        if H[i]==w.max() and (w==H[i]).sum()==1: out.append(i)
    return out
def find_pivot_lows(L, left=5, right=5):
    n=len(L); out=[]
    for i in range(left, n-right):
        w=L[i-left:i+right+1]
        if L[i]==w.min() and (w==L[i]).sum()==1: out.append(i)
    return out


CUP_COLS = ['lch', 'rch', 'cli', 'cup_low', 'lh', 'rh', 'cup_rim', 'depth_pct']

def enumerate_cups(p):
    """Every (left lip, right lip) pair passing the PIT_C cup tests under the WIDEST bounds, in PIT_C's
    order (lch ascending, rch ascending), WITHOUT the one-cup-per-lip break — variants apply that later."""
    H, L = p['High'], p['Low']
    ph = find_pivot_highs(H, PIV, PIV)
    if len(ph) < 2: return []
    out = []
    for a, lch in enumerate(ph[:-1]):
        lh = H[lch]
        for rch in ph[a+1:]:
            dur = rch - lch
            if dur < ENUM_DUR[0]: continue
            if dur > ENUM_DUR[1]: break
            rh = H[rch]
            if abs(rh - lh) / lh > RIM_TOL: continue
            cl = L[lch+1:rch]
            if len(cl) < 5: continue
            cup_low = cl.min(); cli = lch + 1 + int(cl.argmin())
            cup_rim = (lh + rh) / 2; dp = (cup_rim - cup_low) / cup_rim
            if not (ENUM_DEPTH[0] <= dp <= ENUM_DEPTH[1]): continue
            inb = np.flatnonzero(L[lch:rch+1] <= cup_low * (1 + ROUND_BAND))
            if len(inb) < ROUND_MIN: continue
            if inb.max() - inb.min() < ROUND_SPAN: continue
            out.append((lch, rch, cli, cup_low, lh, rh, cup_rim, dp))
    return out
# ---- run ----
tickers = sorted(price_db); CHUNK = 200; t0 = time.time()
for k in range(0, len(tickers), CHUNK):
    path = CACHE/f'cups_{k//CHUNK:03d}.parquet'
    if path.exists(): continue
    rows = []
    for tk in tickers[k:k+CHUNK]:
        rows += [(tk,) + c for c in enumerate_cups(price_db[tk])]
    pd.DataFrame(rows, columns=['ticker'] + CUP_COLS).to_parquet(path)
    print(f'  chunk {k//CHUNK:03d} saved ({min(k+CHUNK, len(tickers))}/{len(tickers)} tickers, {time.time()-t0:.0f}s)')
CUPS = pd.concat([pd.read_parquet(p) for p in sorted(CACHE.glob('cups_*.parquet'))], ignore_index=True)
CUPS_BY_TK = {tk: g.sort_values(['lch', 'rch'], kind='mergesort')[CUP_COLS].itertuples(index=False, name=None)
              for tk, g in CUPS.groupby('ticker')}
CUPS_BY_TK = {tk: list(v) for tk, v in CUPS_BY_TK.items()}
print(f'candidate cups: {len(CUPS):,} on {CUPS.ticker.nunique()} tickers')


In [ ]:
# ===== Cell 3 — Per-variant fire tables (both entry types), A6 rule, trial count  (~2-5 min per variant) =====
def handle_ok(v, cup, hl):
    lch, rch, cli, cup_low, lh, rh, cup_rim, dp = cup
    if hl <= cup_low: return False
    if v['handle'] in ('pitc', 'pitc_upper'):
        retr = (rh - hl) / (cup_rim - cup_low)
        if not (0.10 <= retr <= 0.50): return False
        if v['handle'] == 'pitc_upper' and hl < cup_low + 0.5 * (lh - cup_low): return False
        return True
    if hl < cup_low + 0.5 * (lh - cup_low): return False           # oneil: upper half of the base
    return (rh - hl) / rh <= v['handle_max_drop']                   # oneil: handle drop from the right lip

def prior_up_ok(v, lch, H, L):
    if v['handle'] != 'oneil': return True
    lb = v['prior_lb']
    if lch < lb - 1: return False
    return H[lch] / L[lch-lb+1:lch+1].min() - 1 >= v['prior_up']

def accept(fill, hl, atr):
    """PIT_C's handle-low risk checks — part of the PATTERN (they decide which cup per lip fires)."""
    if np.isnan(atr) or atr <= 0: return None
    sd = fill - hl * 0.999
    if sd <= 0: return None
    if sd < MIN_STOP_ATR * atr: sd = MIN_STOP_ATR * atr
    if sd / fill > MAX_STOP_PCT: return None
    return fill - sd

def close_fire(v, cup, p):
    """PIT_C: first CLOSE > rim after the right lip; handle low = min low strictly between; enter next open."""
    lch, rch, cli, cup_low, lh, rh, cup_rim, dp = cup
    O, L, C = p['Open'], p['Low'], p['Close']; n = len(C); rim = max(lh, rh)
    j = None
    for t in range(rch+1, min(rch+1+HANDLE_WINDOW+CONFIRM_WINDOW_BARS, n)):
        if C[t] > rim: j = t; break
    if j is None: return None
    seg = L[rch+1:j]
    if len(seg) == 0: return None
    h = rch + 1 + int(seg.argmin()); hl = L[h]
    if h - rch > HANDLE_WINDOW or j - h > CONFIRM_WINDOW_BARS: return None
    if not handle_ok(v, cup, hl): return None
    e = j + 1
    if e >= n: return None
    stop = accept(O[e], hl, p['ATR'][rch])
    if stop is None: return None
    return dict(dec=j, e=e, h=h, hl=hl, fill=O[e], pitc_stop=stop, rim=rim)

def buystop_fire(v, cup, p, _leak=False):
    """Buy-stop at the rim. Armed at the close of t-1 from bars <= t-1 (handle low = min L[rch+1..t-1],
    windows, handle rules); fills on the FIRST bar t with High >= rim at max(Open, rim). The first touch
    decides the cup (price has traded through the rim). _leak: NEGATIVE CONTROL that peeks at C[t]."""
    lch, rch, cli, cup_low, lh, rh, cup_rim, dp = cup
    O, H, L, C = p['Open'], p['High'], p['Low'], p['Close']; n = len(H); rim = max(lh, rh)
    for t in range(rch+2, min(rch+1+HANDLE_WINDOW+CONFIRM_WINDOW_BARS, n)):
        if H[t] >= rim:
            if _leak and not (C[t] > rim): return None
            seg = L[rch+1:t]; h = rch + 1 + int(seg.argmin()); hl = L[h]
            if h - rch > HANDLE_WINDOW or t - h > CONFIRM_WINDOW_BARS: return None
            if not handle_ok(v, cup, hl): return None
            fill = max(O[t], rim)
            stop = accept(fill, hl, p['ATR'][rch])
            if stop is None: return None
            return dict(dec=t-1, e=t, h=h, hl=hl, fill=fill, pitc_stop=stop, rim=rim)
    return None

def variant_fires(vname, entry, tk, p, cups, _leak=False):
    v = VARIANTS[vname]; out = []; cur = None; done = False
    for cup in cups:
        lch, rch = cup[0], cup[1]
        if lch != cur: cur, done = lch, False
        if done: continue
        if not (v['dur'][0] <= rch - lch <= v['dur'][1]): continue
        if not (v['depth'][0] <= cup[7] <= v['depth'][1]): continue
        if not prior_up_ok(v, lch, p['High'], p['Low']): continue
        f = close_fire(v, cup, p) if entry == 'next_open' else buystop_fire(v, cup, p, _leak)
        if f is None: continue
        done = True                                                  # one cup per left lip (PIT_C)
        D = p['Date']; S50, S200 = p['S50'], p['S200']
        f.update(ticker=tk, variant=vname, entry_type=entry, lch=lch, rch=rch, cli=cup[2], cup_low=cup[3],
                 lh=cup[4], rh=cup[5], cup_rim=cup[6], depth_pct=cup[7], depth_abs=cup[6]-cup[3],
                 target_price=f['rim'] + (cup[6]-cup[3]), atr_rch=p['ATR'][rch], atr_dec=p['ATR'][f['dec']],
                 entry_date=pd.Timestamp(D[f['e']]), dec_date=pd.Timestamp(D[f['dec']]),
                 handle_low_date=pd.Timestamp(D[f['h']]), rch_date=pd.Timestamp(D[rch]),
                 v2b_dec=bool(S50[f['dec']] > S200[f['dec']]), v2b_entry=bool(S50[f['e']] > S200[f['e']]))
        out.append(f)
    return out

KEY = ['ticker','entry_date','handle_low_date','rch_date','_tgt']
def with_tgt(df): return df.assign(_tgt=df['target_price'].round(6))

def dedupe(fires, pitc_ref=None):
    """PIT_C dedupe: earliest entry per (ticker, handle_low_date). Ties resolved like PIT_C when its saved
    raw-fire file is given (A1 next-open), else deterministically."""
    fires = with_tgt(fires); fires['_pitc'] = False
    if pitc_ref is not None:
        fires['_pitc'] = pd.MultiIndex.from_frame(fires[KEY]).isin(pd.MultiIndex.from_frame(pitc_ref[KEY]))
    fires = fires.sort_values(['entry_date','_pitc','ticker','lch','rch'], ascending=[True, False, True, True, True], kind='mergesort')
    return fires.drop_duplicates(['ticker','handle_low_date']).drop(columns='_pitc').reset_index(drop=True)

def fire_table(vname, entry, pitc_ref=None):
    rows = []
    for tk, p in price_db.items():
        rows += variant_fires(vname, entry, tk, p, CUPS_BY_TK.get(tk, []))
    F = dedupe(pd.DataFrame(rows), pitc_ref)
    return F[F.entry_date <= PURGE_DATE].reset_index(drop=True)            # nothing after the purge goes further
# ---- run ----
raw_ref = HIST_DIR/'cup_handle_fires_15yr_raw_PIT_C.csv'
PITC_RAW = with_tgt(read_is_only(raw_ref, KEY[:-1] + ['target_price'])) if raw_ref.exists() else None
FIRES = {}
for vname in VARIANTS:
    for entry in ENTRIES:
        path = CACHE/f'fires_{vname}_{entry}.parquet'
        if not path.exists():
            t0 = time.time()
            fire_table(vname, entry, PITC_RAW if (vname, entry) == ('A1', 'next_open') else None).to_parquet(path)
            print(f'  {vname} {entry}: saved ({time.time()-t0:.0f}s)')
        FIRES[(vname, entry)] = pd.read_parquet(path)
        F = FIRES[(vname, entry)]
        print(f'{vname} {entry:9s}: {len(F):6d} fires (pre-v2b) | v2b@decision {int(F.v2b_dec.sum()):6d} | IS {int(((F.entry_date >= IS_START) & F.v2b_dec).sum())}')

key = lambda F: set(zip(F.ticker, F.lch, F.rch, F.e))
a1, a6 = key(FIRES[('A1','next_open')]), key(FIRES[('A6','next_open')])
A6_DIFF = len(a1 ^ a6) / max(len(a1), 1)
RUN = [v for v in VARIANTS if v != 'A6'] + (['A6'] if A6_DIFF >= A6_MIN_DIFF else [])
print(f'A6 vs A1 next-open fires differ by {A6_DIFF:.1%} -> A6 {"RUNS" if "A6" in RUN else "DROPPED (not counted as trials)"}')

# stop-distance ORDER for the plateau gate — geometry only, before any exit is simulated
F = FIRES[('A1','next_open')]; F = F[F.v2b_dec & (F.entry_date >= IS_START)]
_med = {'handle': ((F.fill - F.pitc_stop) / F.fill).median(), 'pct7': 0.07, 'pct8': 0.08,
        'atr2': (2 * F.atr_dec / F.fill).median(), 'atr3': (3 * F.atr_dec / F.fill).median()}
STOP_ORDER = sorted(STOPS, key=lambda s: _med[s])
json.dump(dict(STOP_ORDER=STOP_ORDER, medians=_med, RUN=RUN, A6_DIFF=A6_DIFF), open(CACHE/'tierA_decisions.json', 'w'))
N_STAGE2 = len(RUN) * len(ENTRIES)*len(STOPS)*len(TARGETS)*len(TRAILS)*len(TIMES)*len(GATES)
print(f'STOP_ORDER (IS median initial distance): {[(s, round(_med[s], 4)) for s in STOP_ORDER]}')
print(f'TRIAL COUNT: Stage 2 = {N_STAGE2:,} | total for DSR = {STAGE1_TRIALS + N_STAGE2:,}')


In [ ]:
# ===== Cell 4 — A1 PARITY vs PIT_C (PIT_C fills + v2b at entry bar; IS <= purge)  (~3 min) =====
@njit(cache=True)
def _sim_trade(opens,highs,lows,closes, entry_idx, entry_price, stop_price,
               target_price, atr, time_stop_days):
    n=len(opens); end=entry_idx+time_stop_days
    if end>n: end=n
    risk = entry_price-stop_price
    for i in range(entry_idx, end):
        bl=lows[i]; bh=highs[i]
        if bl <= stop_price:
            return i, stop_price, 0, (stop_price-entry_price)/risk
        if bh >= target_price:
            return i, target_price, 1, (target_price-entry_price)/risk
    last=end-1
    return last, closes[last], 4, (closes[last]-entry_price)/risk

# warm up
_sim_trade(np.array([100.,101,102]),np.array([100.,101,102]),np.array([99.,100,101]),
           np.array([100.,101,102]),0,100.,95.,110.,2.,120)

REASON={0:'stop',1:'target',4:'time_stop'}

def parity_trades(F, pitc_v2b_ref):
    """Reproduce PIT_C's trades: v2b at ENTRY bar (PIT_C's one-bar look-ahead), t05, PIT_C stop,
    fill-at-stop/target, 120-bar time stop, non-overlap ordered like PIT_C's v2b file."""
    F = F[F.v2b_entry].reset_index(drop=True); res = []
    for f in F.itertuples():
        p = price_db[f.ticker]
        tgt = f.rim + 0.5 * f.depth_abs
        xi, xpx, rc, R = _sim_trade(p['Open'], p['High'], p['Low'], p['Close'], int(f.e), f.fill, f.pitc_stop,
                                    tgt, f.atr_rch, TIME_STOP_DAYS)
        res.append(dict(exit_date=pd.Timestamp(p['Date'][xi]), R=R, exit_reason=REASON[rc]))
    F = pd.concat([F, pd.DataFrame(res)], axis=1)
    F['_ord'] = np.inf
    if pitc_v2b_ref is not None:
        r = pitc_v2b_ref.copy(); r['_ord_ref'] = np.arange(len(r), dtype=float)
        F = F.merge(r, on=KEY, how='left'); F['_ord'] = F['_ord_ref'].fillna(np.inf); F = F.drop(columns='_ord_ref')
    F = F.sort_values(['ticker','entry_date','_ord','lch','rch'], kind='mergesort').reset_index(drop=True)
    keep = []; last = {}
    for r in F.itertuples():
        pv = last.get(r.ticker)
        if pv is None or r.entry_date > pv: keep.append(True); last[r.ticker] = r.exit_date
        else: keep.append(False)
    F['kept'] = keep
    return F
# ---- run ----
from collections import Counter
rf_path, rt_path = HIST_DIR/'cup_handle_v2b_fires_15yr_PIT_C.csv', HIST_DIR/'trades_t05_15yr_PIT_C.csv'
if rf_path.exists() and rt_path.exists():
    rf = with_tgt(read_is_only(rf_path, KEY[:-1] + ['target_price'])).drop(columns='target_price')
    P = parity_trades(FIRES[('A1','next_open')], rf)
    a = Counter(map(tuple, with_tgt(P)[KEY].values.tolist())); b = Counter(map(tuple, rf[KEY].values.tolist()))
    print(f'A1 parity — v2b fires (full key, multiset): mine {sum(a.values())} | PIT_C {sum(b.values())} | diff {sum((a-b).values()) + sum((b-a).values())}')
    rt = read_is_only(rt_path, ['ticker','entry_date','R'])
    mm = P[P.kept][['ticker','entry_date','R']].merge(rt, on=['ticker','entry_date'], how='outer', suffixes=('','_ref'), indicator=True)
    d = (mm.R - mm.R_ref).abs().max()
    print(f'A1 parity — trades: matched {int((mm._merge=="both").sum())} | only-mine {int((mm._merge=="left_only").sum())} | only-PIT_C {int((mm._merge=="right_only").sum())} | max |R diff| {d:.2e}')
    assert a == b and (mm._merge == 'both').all() and d < 1e-9, 'A1 does not reproduce PIT_C — stop and reconcile'
else:
    print('WARNING: PIT_C outputs not found — A1 parity NOT asserted')


In [ ]:
# ===== Cell 5 — CUT-OFF CAUSALITY TESTS per variant x entry, with negative controls  (~4-8 min per variant) =====
def _trunc(p, last):
    return {k: (v[:last+1].copy() if isinstance(v, np.ndarray) else v) for k, v in p.items()}

def _rederive(q):
    d = pd.DataFrame({'High': q['High'], 'Low': q['Low'], 'Close': q['Close']})
    q['ATR'] = compute_atr(d, ATR_PERIOD).to_numpy(float)
    for k, w in (('S50', STOCK_FAST_SMA), ('S200', STOCK_SLOW_SMA), ('SMA10', 10), ('SMA21', 21), ('SMA50', 50)):
        q[k] = d['Close'].rolling(w).mean().to_numpy(float)
    return q

def _same(a, b):
    if isinstance(a, (bool, np.bool_)) or isinstance(b, (bool, np.bool_)): return bool(a) == bool(b)
    a, b = float(a), float(b)
    if np.isnan(a) and np.isnan(b): return True
    return abs(a - b) <= 1e-9 * max(1.0, abs(a))

CHECK = ['h', 'hl', 'fill', 'pitc_stop', 'rim', 'atr_rch', 'atr_dec', 'v2b_dec', 'target_price']

def regime_slow(d):
    """Regime re-derived from data cut at d: breadth from the panel truncated at d (shared helper),
    rsp_spy_20d from SPY/RSP arrays truncated at d."""
    d = pd.Timestamp(d)
    k_s, k_r = _idx_le(SPY['Date'], d), _idx_le(RSP['Date'], d)
    spyA = dict(Date=SPY['Date'][:k_s+1], Close=SPY['Close'][:k_s+1]); rspA = dict(Date=RSP['Date'][:k_r+1], Close=RSP['Close'][:k_r+1])
    hist = PANEL_CLOSE.loc[:d]
    b, _ = breadth_window(np.ascontiguousarray(hist.to_numpy(float))[-50:])
    return b, mkt_features(d, spyA, rspA)['rsp_spy_20d']

def cutoff_test_fires(vname, entry, F, n=200, seed=0, mode='test'):
    """mode='test' : re-detect on data CUT at the entry bar (next_open: only its OPEN known;
                     buy_stop: only its OPEN and HIGH known) and compare every field + regime.
       mode='ctrl_v2b'   : NEGATIVE CONTROL — compare PIT_C's v2b-at-entry-bar (must mismatch).
       mode='ctrl_close' : NEGATIVE CONTROL — buy-stop arming that peeks at C[t] (must mismatch)."""
    rng = np.random.default_rng(seed); pick = rng.choice(len(F), size=min(n, len(F)), replace=False); bad = []
    for i in pick:
        f = F.iloc[i].to_dict(); tk = f['ticker']; p = price_db[tk]; e = int(f['e'])
        q = _trunc(p, e)
        masked = ('High', 'Low', 'Close', 'Volume') if entry == 'next_open' else ('Low', 'Close', 'Volume')
        for k in masked: q[k][e] = np.nan
        q = _rederive(q)
        g = [x for x in variant_fires(vname, entry, tk, q, enumerate_cups(q), _leak=(mode == 'ctrl_close'))
             if (x['lch'], x['rch'], x['e']) == (f['lch'], f['rch'], e)]
        if len(g) != 1: bad.append((tk, str(f['entry_date'].date()), 'fire not re-detected')); continue
        g = g[0]
        if mode == 'ctrl_v2b': g['v2b_dec'] = bool(q['S50'][e] > q['S200'][e])        # PIT_C's entry-bar v2b
        bad += [(tk, str(f['entry_date'].date()), k) for k in CHECK if not _same(f[k], g[k])]
        if mode == 'test' and f['dec_date'] in REGIME.index:
            b, r = regime_slow(f['dec_date']); row = REGIME.loc[f['dec_date']]
            if not (_same(b, row.breadth_50) and _same(r, row.rsp_spy_20d)): bad.append((tk, str(f['entry_date'].date()), 'regime'))
    return bad
# ---- run ----
PANEL_CLOSE = pd.DataFrame({tk: pd.Series(v['Close'], index=pd.DatetimeIndex(v['Date'])) for tk, v in PANEL_SRC.items()}).reindex(pd.DatetimeIndex(SPY['Date']))
for vname in RUN:
    for entry in ENTRIES:
        path = CACHE/f'causality_{vname}_{entry}.json'
        if path.exists(): print(vname, entry, json.load(open(path))); continue
        F = FIRES[(vname, entry)]; F = F[F.v2b_dec & (F.entry_date >= IS_START)].reset_index(drop=True)
        t0 = time.time()
        ctrl = cutoff_test_fires(vname, entry, F, n=30, seed=1, mode='ctrl_v2b' if entry == 'next_open' else 'ctrl_close')
        assert len(ctrl) > 0, f'NEGATIVE CONTROL PASSED for {vname} {entry} — tester is blind'
        bad = cutoff_test_fires(vname, entry, F, n=200, seed=0)
        res = dict(n=min(200, len(F)), mismatches=len(bad), examples=bad[:10], control_mismatches=len(ctrl), secs=round(time.time()-t0))
        print(vname, entry, res)
        assert len(bad) == 0, f'CAUSALITY FAILED for {vname} {entry} — no Tier B until fixed'
        json.dump(res, open(path, 'w'))


In [ ]:
# ===== Cell 6 — Tier B: exit grid + per-config non-overlap (numba)  (~3-10 min per variant; resumable) =====
@njit(cache=True)
def _sim_path(O, H, L, C, ATR, TR, e, n, fill, stop, target, trail, T, chand):
    """One trade, REALISTIC fills, bar order: stop (low) -> target (high) -> time (close of last bar) ->
    trailing signal at the close, exit at the NEXT open. Reasons: 0 stop 1 target 2 time 3 end-of-data 4 trail."""
    end = e + T; last = (end if end < n else n) - 1; hh = -1e300
    for i in range(e, last + 1):
        if L[i] <= stop:
            return i, (stop if i == e else min(O[i], stop)), 0
        if H[i] >= target:
            return i, (max(fill, target) if i == e else max(O[i], target)), 1
        if i == last:
            return i, C[i], (3 if end > n else 2)
        if trail == 4:
            if H[i] > hh: hh = H[i]
            if C[i] < hh - chand * ATR[i]: return i + 1, O[i + 1], 4
        elif trail > 0:
            if C[i] < TR[i]: return i + 1, O[i + 1], 4
    return last, C[last], 2

@njit(parallel=True, cache=True)
def run_grid(Of, Hf, Lf, Cf, ATRf, S10f, S21f, S50f, DAYf, off, nbar, nfull,
             f_tk, f_e, f_fill, f_edate, f_regime, f_month, f_year, f_in_is, STOPm, SKIPm, TGTm,
             c_stop, c_tgt, c_trail, c_time, chand, cost, n_months, n_years):
    nc = len(c_stop); nf = len(f_tk)
    N = np.zeros((nc, 2)); NWIN = np.zeros((nc, 2)); WR_ = np.zeros((nc, 2)); LR_ = np.zeros((nc, 2))
    WM = np.zeros((nc, 2, n_months)); LM = np.zeros((nc, 2, n_months)); CM = np.zeros((nc, 2, n_months))
    RY = np.zeros((nc, 2, n_years)); CY = np.zeros((nc, 2, n_years)); SRET = np.zeros((nc, 2)); SHELD = np.zeros((nc, 2))
    NCLIP = np.zeros((nc, 2))
    for c in prange(nc):
        s = c_stop[c]; g = c_tgt[c]; tr = c_trail[c]; T = c_time[c]
        for gate in range(2):
            last_tk = -1; last_exit = -(1 << 62); blocked = False
            for k in range(nf):
                tk = f_tk[k]
                if tk != last_tk:
                    last_tk = tk; last_exit = -(1 << 62); blocked = False
                if blocked: continue
                if gate == 1 and not f_regime[k]: continue
                if SKIPm[s, k]: continue
                if f_edate[k] <= last_exit: continue
                o = off[tk]; n = nbar[tk]; stop = STOPm[s, k]; fill = f_fill[k]
                TR = S10f[o:o+n] if tr == 1 else (S21f[o:o+n] if tr == 2 else S50f[o:o+n])
                xi, px, rc = _sim_path(Of[o:o+n], Hf[o:o+n], Lf[o:o+n], Cf[o:o+n], ATRf[o:o+n], TR,
                                       f_e[k], n, fill, stop, TGTm[g, k], tr, T, chand)
                if rc == 3 and nfull[tk] > n:          # time stop clipped by the cut: drop it and the ticker's later entries
                    blocked = True; NCLIP[c, gate] += 1; continue
                last_exit = DAYf[o + xi]
                if not f_in_is[k]: continue              # pre-2011 entries shape the overlap chain only
                risk = fill - stop; r = (px - fill) / risk; rc_ = r - cost * fill / risk
                ret = px / fill - 1.0 - cost
                N[c, gate] += 1; SRET[c, gate] += ret; SHELD[c, gate] += xi - f_e[k] + 1
                if r > 0: NWIN[c, gate] += 1; WR_[c, gate] += r
                else: LR_[c, gate] -= r
                m = f_month[k]; y = f_year[k]
                if rc_ > 0: WM[c, gate, m] += rc_
                else: LM[c, gate, m] -= rc_
                CM[c, gate, m] += 1; RY[c, gate, y] += ret; CY[c, gate, y] += 1
    return N, NWIN, WR_, LR_, WM, LM, CM, RY, CY, SRET, SHELD, NCLIP

EXIT_CFGS = list(itertools.product(range(len(STOPS)), range(len(TARGETS)), range(len(TRAILS)), TIMES))
TRAIL_CODE = {'none': 0, 'sma10': 1, 'sma21': 2, 'sma50': 3, 'chand3': 4}

def flat_prices(tickers):
    off, nb, nfl = [], [], []; cols = {k: [] for k in ['Open','High','Low','Close','ATR','SMA10','SMA21','SMA50']}; days = []; o = 0
    for tk in tickers:
        p = price_db[tk]; off.append(o); nb.append(len(p['Close'])); nfl.append(N_FULL[tk]); o += len(p['Close'])
        for k in cols: cols[k].append(p[k])
        days.append(p['Date'].astype('datetime64[D]').astype(np.int64))
    cat = {k: np.concatenate(v).astype(np.float64) for k, v in cols.items()}
    return cat, np.concatenate(days), np.array(off, np.int64), np.array(nb, np.int64), np.array(nfl, np.int64)

def fire_arrays(F, tk_index):
    """Fires sorted (ticker, entry_date, lch, rch) — the per-config non-overlap order."""
    F = F.sort_values(['ticker','entry_date','lch','rch'], kind='mergesort').reset_index(drop=True)
    fill = F.fill.to_numpy(float); atr = F.atr_dec.to_numpy(float)
    st = np.vstack([F.pitc_stop.to_numpy(float), fill*0.93, fill*0.92, fill - 2*atr, fill - 3*atr])
    skip = ~np.isfinite(st) | (st >= fill) | ((fill - st) / fill > MAX_STOP_PCT + 1e-12)
    rim, dep = F.rim.to_numpy(float), F.depth_abs.to_numpy(float)
    tg = np.vstack([rim+0.5*dep, rim+0.75*dep, rim+1.0*dep, rim+1.5*dep, fill*1.15, fill*1.20, fill*1.25, np.full(len(F), np.inf)])
    reg = REGIME['on'].reindex(pd.DatetimeIndex(F.dec_date)).fillna(False).to_numpy(bool)
    ed = F.entry_date
    in_is = ((ed >= IS_START) & (ed <= PURGE_DATE)).to_numpy()
    month = np.where(in_is, (ed.dt.year - IS_YEARS[0]) * 12 + ed.dt.month - 1, 0).astype(np.int64)
    year = np.where(in_is, ed.dt.year - IS_YEARS[0], 0).astype(np.int64)
    return dict(F=F, tk=F.ticker.map(tk_index).to_numpy(np.int64), e=F.e.to_numpy(np.int64), fill=fill,
                edate=ed.values.astype('datetime64[D]').astype(np.int64), regime=reg, month=month, year=year,
                in_is=in_is, STOP=st, SKIP=skip, TGT=tg)

def run_variant(vname, entry, P, A):
    cs = np.array([c[0] for c in EXIT_CFGS], np.int64); ct = np.array([c[1] for c in EXIT_CFGS], np.int64)
    ctr = np.array([TRAIL_CODE[TRAILS[c[2]]] for c in EXIT_CFGS], np.int64); cti = np.array([c[3] for c in EXIT_CFGS], np.int64)
    cat, days, off, nb, nfl = P
    return run_grid(cat['Open'], cat['High'], cat['Low'], cat['Close'], cat['ATR'], cat['SMA10'], cat['SMA21'], cat['SMA50'],
                    days, off, nb, nfl, A['tk'], A['e'], A['fill'], A['edate'], A['regime'], A['month'], A['year'], A['in_is'],
                    A['STOP'], A['SKIP'], A['TGT'], cs, ct, ctr, cti, CHAND_MULT, COST_PCT, N_MONTHS, len(IS_YEARS))

def sim_cutoff_test(A, P, n=200, seed=0, leak=False):
    """Exit causality: re-simulate sampled (fire, config) pairs on prices CUT at the exit bar, indicators
    re-derived from the cut data. Stop / target / time exits: bar xi fully known. Trailing exits (filled at
    bar xi's OPEN): bar xi's high/low/close masked. Exits by end-of-data are not sampled.
    leak=True: NEGATIVE CONTROL — an SMA trailing signal that peeks at the next close; must mismatch."""
    cat, days, off, nb, nfl = P; rng = np.random.default_rng(seed); bad = 0; tried = 0; guard = 0
    names = ['Open','High','Low','Close','ATR','SMA10','SMA21','SMA50']
    def sim(a, k, s, g, trn, T, nn):
        TR = a['SMA10'] if trn == 1 else (a['SMA21'] if trn == 2 else a['SMA50'])
        if leak:
            TR = TR.copy(); nxt = np.r_[a['Close'][1:], np.nan]
            TR = np.where(nxt < a['Close'], np.inf, TR)                     # peeks at C[i+1]
        return _sim_path(a['Open'], a['High'], a['Low'], a['Close'], a['ATR'], TR, A['e'][k], nn,
                         A['fill'][k], A['STOP'][s, k], A['TGT'][g, k], trn, T, CHAND_MULT)
    while tried < n and guard < 100 * n:
        guard += 1
        k = int(rng.integers(len(A['tk']))); s, g, t_, T = EXIT_CFGS[int(rng.integers(len(EXIT_CFGS)))]
        trn = TRAIL_CODE[TRAILS[t_]]
        if A['SKIP'][s, k] or (leak and trn not in (1, 2, 3)): continue
        tk = A['tk'][k]; o, n_ = off[tk], nb[tk]
        arr = {nm: cat[nm][o:o+n_].copy() for nm in names}
        xi, px, rc = sim(arr, k, s, g, trn, T, n_)
        if rc == 3 or (leak and rc != 4): continue
        tried += 1
        q = {nm: v[:xi+1].copy() for nm, v in arr.items()}
        if rc == 4:
            for nm in ('High', 'Low', 'Close'): q[nm][xi] = np.nan
        d = pd.DataFrame({'High': q['High'], 'Low': q['Low'], 'Close': q['Close']})
        q['ATR'] = compute_atr(d, ATR_PERIOD).to_numpy(float)
        for w in (10, 21, 50): q[f'SMA{w}'] = d['Close'].rolling(w).mean().to_numpy(float)
        xi2, px2, rc2 = sim(q, k, s, g, trn, T, xi + 1)
        if not (xi2 == xi and rc2 == rc and _same(px2, px)): bad += 1
    return bad
# ---- run ----
TK_LIST = sorted(price_db); TK_INDEX = {tk: i for i, tk in enumerate(TK_LIST)}
PRICES = flat_prices(TK_LIST); ARR = {}
for vname in RUN:
    for entry in ENTRIES:
        path = CACHE/f'grid_{vname}_{entry}.npz'
        F = FIRES[(vname, entry)]; ARR[(vname, entry)] = A = fire_arrays(F[F.v2b_dec], TK_INDEX)
        if path.exists(): continue
        if (vname, entry) == (RUN[0], 'next_open'):
            ctrl = sim_cutoff_test(A, PRICES, n=40, seed=1, leak=True)
            assert ctrl > 0, 'exit NEGATIVE CONTROL passed — exit tester is blind'
            bad = sim_cutoff_test(A, PRICES, n=200, seed=0)
            print(f'exit cut-off causality: 200 (fire, config) pairs, {bad} mismatches | negative control {ctrl}/40 caught')
            assert bad == 0, 'EXIT CAUSALITY FAILED'
        t0 = time.time()
        out = run_variant(vname, entry, PRICES, A)
        np.savez_compressed(path, *out)
        print(f'  {vname} {entry}: {len(EXIT_CFGS)} exit configs x 2 gates on {len(A["tk"])} fires ({time.time()-t0:.0f}s)')


In [ ]:
# ===== Cell 7 — Metrics, gates 1-3 (CI only where it can matter), trial log  (~10-20 min) =====
OUT_NAMES = ['N','NWIN','WR_','LR_','WM','LM','CM','RY','CY','SRET','SHELD','NCLIP']
def pf(w, l): return w / l if l > 0 else (np.inf if w > 0 else np.nan)

def month_block_ci(Wm, Lm, level, reps, seed):
    m = (Wm + Lm) > 0
    W, L = Wm[m], Lm[m]
    if len(W) == 0: return np.nan, np.nan
    idx = np.random.default_rng(seed).integers(0, len(W), (reps, len(W)))
    w, l = W[idx].sum(1), L[idx].sum(1)
    pfs = np.where(l > 0, w / np.where(l > 0, l, 1), np.inf); a = (1 - level) / 2
    return float(np.quantile(pfs, a)), float(np.quantile(pfs, 1 - a))

def config_id(row):
    key = json.dumps([SWEEP_ID, row['variant'], row['entry'], row['stop'], row['target'], row['trail'], int(row['time']),
                      row['gate'], str(CUT_DATE.date()), str(PURGE_DATE.date()), COST_PCT, GATE], sort_keys=True)
    return hashlib.sha1(key.encode()).hexdigest()[:12]

def build_table(grids):
    rows, series, yret, ycnt, wms, lms, cms = [], [], [], [], [], [], []
    for (vname, entry), out in grids.items():
        G = dict(zip(OUT_NAMES, out))
        for c, (s, g, t, T) in enumerate(EXIT_CFGS):
            for gi, gate in enumerate(GATES):
                n = G['N'][c, gi]; Wm, Lm = G['WM'][c, gi], G['LM'][c, gi]
                rows.append(dict(variant=vname, entry=entry, stop=STOPS[s], target=TARGETS[g], trail=TRAILS[t], time=T, gate=gate,
                                 n=int(n), WR=G['NWIN'][c, gi] / n if n else np.nan, PF=pf(G['WR_'][c, gi], G['LR_'][c, gi]),
                                 PF_cost=pf(Wm.sum(), Lm.sum()), avgR=(G['WR_'][c, gi] - G['LR_'][c, gi]) / n if n else np.nan,
                                 avgR_cost=(Wm.sum() - Lm.sum()) / n if n else np.nan,
                                 avg_ret=G['SRET'][c, gi] / n if n else np.nan, avg_bars=G['SHELD'][c, gi] / n if n else np.nan,
                                 ret_per_bar=G['SRET'][c, gi] / G['SHELD'][c, gi] if G['SHELD'][c, gi] else np.nan,
                                 n_clipped=int(G['NCLIP'][c, gi])))
                series.append(Wm - Lm); yret.append(G['RY'][c, gi]); ycnt.append(G['CY'][c, gi])
                wms.append(Wm); lms.append(Lm); cms.append(G['CM'][c, gi])
    T_ = pd.DataFrame(rows)
    return T_, np.array(series), np.array(yret), np.array(ycnt), np.array(wms), np.array(lms), np.array(cms)

def gates_1_3(T_, S, YR, YC, WMs, LMs):
    """S: monthly net cost-R (configs x 156); YR/YC: yearly % return sums / counts."""
    yavg = np.where(YC > 0, YR / np.where(YC > 0, YC, 1), 0.0)              # empty year = 0% (cash)
    ci = (T_[['variant','entry','stop','target','trail','time','gate']] == pd.Series(CONTROL)).all(axis=1)
    assert ci.sum() == 1, 'control config missing'; base = yavg[ci.to_numpy()][0]
    T_['years_beat'] = (yavg > base).sum(axis=1)
    ynet = S.reshape(len(S), len(IS_YEARS), 12).sum(axis=2); tot = ynet.sum(axis=1)
    T_['max_year_share'] = np.where(tot > 0, ynet.max(axis=1) / np.where(tot > 0, tot, 1), np.inf)
    # plateau
    idx = {tuple(r): i for i, r in enumerate(T_[['variant','entry','stop','target','trail','time','gate']].itertuples(index=False, name=None))}
    pfc = T_.PF_cost.fillna(0).to_numpy()
    so = STOP_ORDER
    def nbrs(r):
        v, en, st, tg, tr, ti, ga = r; out = []
        out += [(v, x, st, tg, tr, ti, ga) for x in ENTRIES if x != en]
        out += [(v, en, st, tg, tr, ti, x) for x in GATES if x != ga]
        k = so.index(st); out += [(v, en, so[j], tg, tr, ti, ga) for j in (k-1, k+1) if 0 <= j < len(so)]
        out += [(v, en, st, x, tr, ti, ga) for x in TARGET_NBRS[tg]]
        out += [(v, en, st, tg, x, ti, ga) for x in TRAIL_NBRS[tr]]
        out += [(v, en, st, tg, tr, x, ga) for x in TIME_NBRS[ti]]
        return [idx[o] for o in out]
    T_['plateau_med'] = [float(np.median(pfc[nbrs(r)])) for r in T_[['variant','entry','stop','target','trail','time','gate']].itertuples(index=False, name=None)]
    g1p = (T_.PF_cost >= GATE['min_pf_cost']) & (T_.n >= GATE['min_n'])
    g2 = (T_.years_beat >= GATE['min_years_beat']) & (T_.max_year_share <= GATE['max_year_share'])
    g3 = T_.plateau_med >= GATE['plateau_median_pf']
    need_ci = ((~g1p).astype(int) + (~g2).astype(int) + (~g3).astype(int)) <= 1
    T_['ci_lo'] = np.nan; T_['ci_hi'] = np.nan
    for i in np.flatnonzero(need_ci.to_numpy()):
        lo, hi = month_block_ci(WMs[i], LMs[i], GATE['ci_level'], GATE['boot_reps'], int(T_.config_id.iat[i][:8], 16))
        T_.iat[i, T_.columns.get_loc('ci_lo')] = lo; T_.iat[i, T_.columns.get_loc('ci_hi')] = hi
    T_['gate1'] = g1p & (T_.ci_lo > GATE['min_ci_lo']); T_['gate2'] = g2; T_['gate3'] = g3
    return T_

def append_log(path, df, key='config_id'):
    if path.exists():
        old = pd.read_csv(path, dtype={key: str}); new = df[~df[key].isin(old[key])]
        pd.concat([old, new], ignore_index=True).to_csv(path, index=False); return len(new), len(old) + len(new)
    df.to_csv(path, index=False); return len(df), len(df)
# ---- run ----
GRIDS = {(v, en): [np.load(CACHE/f'grid_{v}_{en}.npz')[f'arr_{i}'] for i in range(len(OUT_NAMES))] for v in RUN for en in ENTRIES}
CFG, S, YR, YC, WMs, LMs, CMs = build_table(GRIDS)
CFG['config_id'] = [config_id(r) for r in CFG.to_dict('records')]
CFG = gates_1_3(CFG, S, YR, YC, WMs, LMs)
print(f'{len(CFG):,} configs | gate1 {int(CFG.gate1.sum())} | gate2 {int(CFG.gate2.sum())} | gate3 {int(CFG.gate3.sum())} | CI computed for {int(CFG.ci_lo.notna().sum())}')
print(f'clipped-by-cut trades dropped (max over configs): {int(CFG.n_clipped.max())}')
CFG.to_parquet(CACHE/'configs_g123.parquet'); np.savez_compressed(CACHE/'monthly.npz', S=S, WM=WMs, LM=LMs, CM=CMs)


In [ ]:
# ===== Cell 8 — PBO (CPCV over years), effective N, DSR — gates 4-5  (~5-15 min) =====
from scipy.stats import norm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

def pbo_cpcv(Wy, Ly, Ny, n_test, min_train, seed=0):
    """Wy/Ly/Ny: configs x years cost-R wins / losses / trade counts. All C(G, n_test) splits."""
    G = Wy.shape[1]; lam = []
    for test in itertools.combinations(range(G), n_test):
        te = np.zeros(G, bool); te[list(test)] = True; tr = ~te
        wtr, ltr, ntr = Wy[:, tr].sum(1), Ly[:, tr].sum(1), Ny[:, tr].sum(1)
        pf_tr = np.where(ltr > 0, wtr / np.where(ltr > 0, ltr, 1), np.where(wtr > 0, np.inf, np.nan))
        elig = (ntr >= min_train) & ~np.isnan(pf_tr)
        if not elig.any(): continue
        best = np.flatnonzero(elig)[np.argmax(pf_tr[elig])]
        wte, lte, nte = Wy[:, te].sum(1), Ly[:, te].sum(1), Ny[:, te].sum(1)
        pf_te = np.where(lte > 0, wte / np.where(lte > 0, lte, 1), np.where(wte > 0, np.inf, np.nan))
        ok = (nte > 0) & ~np.isnan(pf_te)
        if not ok[best]: lam.append(-np.inf); continue
        vals = pf_te[ok]; r = (vals < pf_te[best]).sum() + 0.5 * ((vals == pf_te[best]).sum() - 1) + 1
        w = r / (ok.sum() + 1); lam.append(np.log(w / (1 - w)))
    lam = np.array(lam)
    return float((lam <= 0).mean()), lam

def effective_n(series):
    """Clusters of trials by 1 - Pearson rho (average linkage, cut at EFFN_CFG['threshold']).
    Zero-variance series are their own clusters."""
    X = np.asarray(series, float); sd = X.std(axis=1); z = sd > 0
    n_zero = int((~z).sum())
    if z.sum() < 2: return int(z.sum()) + n_zero
    D = np.clip(pdist(X[z], 'correlation'), 0, 2)
    lab = fcluster(linkage(D, method='average'), t=EFFN_CFG['threshold'], criterion='distance')
    return int(lab.max()) + n_zero

def dsr(series, N, var_sr):
    X = np.asarray(series, float); T = X.shape[1]
    mu, sd = X.mean(1), X.std(1, ddof=1)
    sr = np.where(sd > 0, mu / np.where(sd > 0, sd, 1), 0.0)
    c = X - mu[:, None]; m2 = (c**2).mean(1)
    skew = np.where(m2 > 0, (c**3).mean(1) / np.where(m2 > 0, m2, 1)**1.5, 0.0)
    kurt = np.where(m2 > 0, (c**4).mean(1) / np.where(m2 > 0, m2, 1)**2, 3.0)
    g = 0.5772156649
    sr0 = np.sqrt(var_sr) * ((1 - g) * norm.ppf(1 - 1/N) + g * norm.ppf(1 - 1/(N * np.e)))
    den = np.sqrt(np.maximum(1 - skew * sr + (kurt - 1) / 4 * sr**2, 1e-12))
    return norm.cdf((sr - sr0) * np.sqrt(T - 1) / den), sr, sr0
# ---- run ----
CFG = pd.read_parquet(CACHE/'configs_g123.parquet'); _M = np.load(CACHE/'monthly.npz')
S, WMs, LMs, CMs = _M['S'], _M['WM'], _M['LM'], _M['CM']
yr = lambda X: X.reshape(len(X), len(IS_YEARS), 12).sum(2)
Wy, Ly, Ny = yr(WMs), yr(LMs), yr(CMs)
t0 = time.time(); PBO, LAM = pbo_cpcv(Wy, Ly, Ny, PBO_CFG['n_test'], PBO_CFG['min_n_train'])
print(f'PBO (CPCV, {len(LAM)} splits) = {PBO:.3f}  -> gate 4 {"PASS" if PBO < GATE["max_pbo"] else "FAIL"} ({time.time()-t0:.0f}s)')
# Stage 1 trials' monthly series on the same 156 months (their R costs were 0.05R per trade)
s1 = pd.read_csv(S1_DIR/'sweep_returns_PIT_C.csv', dtype={'config_id': str})
s1log = pd.read_csv(S1_DIR/'sweep_log_PIT_C.csv', dtype={'config_id': str})
s1ids = s1log[(s1log.kind == 'trial') & (s1log.sweep_id == 'PIT_C_single_factor_v1')].config_id
s1 = s1[s1.config_id.isin(s1ids)].copy()
s1['mi'] = [(int(m[:4]) - IS_YEARS[0]) * 12 + int(m[5:7]) - 1 for m in s1.month]
s1 = s1[(s1.mi >= 0) & (s1.mi < N_MONTHS)]
S1M = np.zeros((len(s1ids), N_MONTHS)); pos = {c: i for i, c in enumerate(s1ids)}
for r in s1.itertuples(): S1M[pos[r.config_id], r.mi] = r.sumR - 0.05 * r.n
t0 = time.time(); N_EFF = effective_n(np.vstack([S1M, S])); N_RAW = STAGE1_TRIALS + len(CFG)
print(f'effective N = {N_EFF:,} of raw {N_RAW:,} trials ({time.time()-t0:.0f}s)')
mu, sd = S.mean(1), S.std(1, ddof=1); VAR_SR = float(np.var(np.where(sd > 0, mu / np.where(sd > 0, sd, 1), 0.0)))
CFG['DSR'], CFG['SR_m'], SR0 = dsr(S, N_EFF, VAR_SR); CFG['DSR_rawN'], _, SR0_raw = dsr(S, N_RAW, VAR_SR)
CFG['gate4'] = PBO < GATE['max_pbo']; CFG['gate5'] = CFG.DSR > GATE['min_dsr']
print(f'SR0 (effective N) = {SR0:.4f} monthly | SR0 (raw N) = {SR0_raw:.4f} | Var[SR] = {VAR_SR:.5f}')
CFG.to_parquet(CACHE/'configs_all_gates.parquet')
log_rows = CFG.assign(kind='trial', sweep_id=SWEEP_ID, feature=CFG.variant, rule=CFG.entry+'|'+CFG.stop+'|'+CFG.target+'|'+CFG.trail+'|'+CFG.time.astype(str)+'|'+CFG.gate,
                      PBO_global=PBO, N_eff=N_EFF, N_raw=N_RAW, logged_at=pd.Timestamp.now(tz='UTC').isoformat(), built_from_commit=BUILT_FROM_COMMIT)
a, b = append_log(S1_DIR/'sweep_log_PIT_C.csv', log_rows)
mr = [dict(config_id=cid, month=f'{IS_YEARS[0] + m // 12}-{m % 12 + 1:02d}', n=int(c), sumR=float(s), cost_basis='price_0.15pct')
      for cid, srow, crow in zip(CFG.config_id, S, CMs) for m, (s, c) in enumerate(zip(srow, crow)) if c > 0]
mr = pd.DataFrame(mr); mr['key'] = mr.config_id + '|' + mr.month
ra, rb = append_log(S1_DIR/'sweep_returns_PIT_C.csv', mr, key='key')
print(f'trial log: +{a} Stage 2 configs (cumulative rows {b}) | monthly rows +{ra}')


In [ ]:
# ===== Cell 9 — Survivors, near misses, heat maps  (~1 min) =====
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
GATE_COLS = ['gate1', 'gate2', 'gate3', 'gate4', 'gate5']
DIV = LinearSegmentedColormap.from_list('pf_div', ['#e34948', '#f0efec', '#2a78d6'])   # red <1 | gray 1 | blue >1
INK, SURF = '#0b0b0b', '#fcfcfb'

def heatmap(ax, M, N, title):
    L = np.log(np.clip(np.nan_to_num(M, nan=1.0, posinf=4.0), 0.25, 4.0))
    nrm = TwoSlopeNorm(vmin=np.log(0.5), vcenter=0.0, vmax=np.log(2.0))
    ax.imshow(L, cmap=DIV, norm=nrm, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            r, g_, b, _ = DIV(nrm(L[i, j])); dark = (0.2126*r + 0.7152*g_ + 0.0722*b) < 0.45   # ink by background
            ink, ink2 = ('#ffffff', '#f0efec') if dark else (INK, '#52514e')
            v = M[i, j]; lab = '—' if np.isnan(v) else ('inf' if np.isinf(v) else f'{v:.2f}')
            ax.text(j, i - 0.12, lab, ha='center', va='center', fontsize=8.5, color=ink)
            ax.text(j, i + 0.22, f'n={int(N[i, j])}', ha='center', va='center', fontsize=6.5, color=ink2)
    ax.set_xticks(range(len(TARGETS)), TARGETS, fontsize=8); ax.set_yticks(range(len(STOP_ORDER)), STOP_ORDER, fontsize=8)
    ax.set_xlabel('target', fontsize=8, color='#52514e'); ax.set_ylabel('stop (ordered by distance)', fontsize=8, color='#52514e')
    ax.set_title(title, fontsize=9, color=INK, loc='left')
    for s in ax.spines.values(): s.set_visible(False)
# ---- run ----
CFG = pd.read_parquet(CACHE/'configs_all_gates.parquet')
CFG['n_fail'] = (~CFG[GATE_COLS]).sum(axis=1)
print('GATES:', json.dumps(GATE))
print('fail counts per gate:', {g: int((~CFG[g]).sum()) for g in GATE_COLS})
cols = ['variant','entry','stop','target','trail','time','gate','n','PF_cost','ci_lo','years_beat','max_year_share','plateau_med','DSR','DSR_rawN','avg_ret','avg_bars']
SURV = CFG[CFG.n_fail == 0].sort_values('DSR', ascending=False)
print(f'\n=== SURVIVORS (all 5 gates): {len(SURV)} ===')
print(SURV[cols].head(20).to_string(index=False, float_format=lambda v: f'{v:.3f}') if len(SURV) else '  none')
NEAR = CFG[CFG.n_fail == 1].copy(); NEAR['failed'] = NEAR[GATE_COLS].idxmin(axis=1)
print(f'\n=== NEAR MISSES (fail exactly one gate): {len(NEAR)} ===')
print(NEAR.sort_values('DSR', ascending=False)[cols + ['failed']].head(25).to_string(index=False, float_format=lambda v: f'{v:.3f}') if len(NEAR) else '  none')
json.dump(SURV.head(3)[['config_id','variant','entry','stop','target','trail','time','gate','DSR']].to_dict('records'),
          open(OUT_DIR/'stage2_survivors.json', 'w'), indent=1, default=str)
CFG.to_csv(OUT_DIR/'stage2_configs.csv', index=False)

ctl = dict(entry='next_open', trail='none', time=120, gate='off')
for vname in RUN:
    C = CFG[CFG.variant == vname]
    A = C[(C.entry == ctl['entry']) & (C.trail == ctl['trail']) & (C.time == ctl['time']) & (C.gate == ctl['gate'])]
    M1 = A.pivot(index='stop', columns='target', values='PF_cost').reindex(index=STOP_ORDER, columns=TARGETS).to_numpy(float)
    N1 = A.pivot(index='stop', columns='target', values='n').reindex(index=STOP_ORDER, columns=TARGETS).to_numpy(float)
    B = C.groupby(['stop','target']).agg(PF_cost=('PF_cost', lambda s: np.nanmedian(s.replace(np.inf, np.nan))), n=('n', 'median')).reset_index()
    M2 = B.pivot(index='stop', columns='target', values='PF_cost').reindex(index=STOP_ORDER, columns=TARGETS).to_numpy(float)
    N2 = B.pivot(index='stop', columns='target', values='n').reindex(index=STOP_ORDER, columns=TARGETS).to_numpy(float)
    fig, axs = plt.subplots(1, 2, figsize=(13, 3.6), facecolor=SURF)
    heatmap(axs[0], M1, N1, f'{vname} PF after cost — next-open, no trail, 120 bars, gate off')
    heatmap(axs[1], M2, N2, f'{vname} median PF after cost over entry x trail x time x gate')
    fig.suptitle('Blue: PF > 1 · gray: PF = 1 · red: PF < 1 (log scale, clipped at 0.5 / 2.0). IS 2011-2023 only.', fontsize=8, color='#52514e', x=0.01, ha='left')
    fig.tight_layout(); fig.savefig(OUT_DIR/f'heatmap_{vname}.png', dpi=140, facecolor=SURF); plt.show()
print('saved:', sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
